# ConforNets Demo

Companion notebook to the `Demo` section in `README.md`. Run the three demo commands first (`preprocess`, `run_diversity`, `run_transfer`) so that `./output/demo/{diversity,transfer}/` is populated, then step through the cells below.

- Evaluates both runs against `repo/toy_assets/toy_benchmark` reference conformations.
- For each reference `i` / `j`, shows the single sample closest to that reference, Kabsch-aligned before py3Dmol renders it.

In [20]:
import subprocess
import sys
import tempfile
from pathlib import Path

import biotite.structure.io.pdb as pdb
import numpy as np
import py3Dmol

from confornet.utils.align import get_sequence, sequence_align
from confornet.utils.io import load_structure

REPO = Path("..").resolve()  # run notebook from repo/
ASSETS = REPO / "repo" / "toy_assets"
BENCH = "toy_benchmark"
DIV_OUT = REPO / "output" / "demo" / "diversity"
XFER_OUT = REPO / "output" / "demo" / "transfer"
REF_DIR = ASSETS / BENCH / "test_cases"
print("repo:", REPO)
print("diversity outputs:", DIV_OUT, "exists" if DIV_OUT.exists() else "MISSING")
print("transfer outputs:", XFER_OUT, "exists" if XFER_OUT.exists() else "MISSING")

repo: /global/u1/m/ml5045/confornet
diversity outputs: /global/u1/m/ml5045/confornet/output/demo/diversity exists
transfer outputs: /global/u1/m/ml5045/confornet/output/demo/transfer exists


In [21]:
def _aligned_pdb_text(mobile_path: Path, anchor) -> str:
    """Kabsch-align mobile onto anchor via sequence-matched CAs; return PDB text."""
    mobile = load_structure(mobile_path)
    matches = sequence_align(get_sequence(mobile), get_sequence(anchor))
    m_ca = mobile[mobile.atom_name == "CA"].coord
    a_ca = anchor[anchor.atom_name == "CA"].coord
    m_pts = np.stack([m_ca[i] for i, _ in matches])
    a_pts = np.stack([a_ca[j] for _, j in matches])
    mc, ac = m_pts.mean(0), a_pts.mean(0)
    U, _, Vt = np.linalg.svd((a_pts - ac).T @ (m_pts - mc))
    F = np.eye(3); F[2, 2] = np.sign(np.linalg.det(U @ Vt))
    R = U @ F @ Vt
    mobile.coord = ((mobile.coord - mc) @ R.T + ac).astype(np.float32)
    with tempfile.NamedTemporaryFile(suffix=".pdb", mode="w+", delete=True) as f:
        pf = pdb.PDBFile()
        pf.set_structure(mobile)
        pf.write(f.name)
        return Path(f.name).read_text()


def show_best(tc_results, ref_idx: int, ref_color: str, sample_color: str,
              width: int = 500, height: int = 500):
    """Render ref[ref_idx] + the sample with the smallest RMSD to it."""
    ref_name = tc_results.references[ref_idx]
    best = int(np.argmin(tc_results.rmsd[:, ref_idx]))
    best_rmsd = float(tc_results.rmsd[best, ref_idx])
    sample_path = Path(tc_results.samples[best].path)

    anchor_path = REF_DIR / tc_results.test_case / "reference" / f"{ref_name}.pdb"
    if not anchor_path.exists():
        anchor_path = anchor_path.with_suffix(".cif")
    anchor = load_structure(anchor_path)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(anchor_path.read_text(), anchor_path.suffix.lstrip("."))
    view.setStyle({"model": -1}, {"cartoon": {"color": ref_color, "opacity": 0.9}})
    view.addModel(_aligned_pdb_text(sample_path, anchor), "pdb")
    view.setStyle({"model": -1}, {"cartoon": {"color": sample_color, "opacity": 0.7}})
    view.zoomTo()
    print(f"{tc_results.test_case} / {ref_name}: best sample {sample_path.name} (RMSD {best_rmsd:.2f} \u00c5)")
    return view

## 1. Evaluate the diversity run

Shells out to `scripts/evaluate.py`, which walks the output tree, computes per-sample RMSDs to every reference via USAlign, and pickles an `EvalResults`.

In [22]:
def run_eval(samples_dir: Path, mode: str, output_pkl: Path) -> None:
    subprocess.run([
        sys.executable, "scripts/evaluate.py",
        "--samples-dir", str(samples_dir / BENCH),
        "--benchmark", BENCH,
        "--assets-dir", str(ASSETS),
        "--mode", mode,
        "--output", str(output_pkl),
    ], cwd=REPO, check=True)

def load_results(pkl: Path):
    from confornet.eval.results import EvalResults
    return EvalResults.load(pkl)

div_pkl = DIV_OUT / "results.pkl"
run_eval(DIV_OUT, "diversity", div_pkl)
div_results = load_results(div_pkl)
for tc, r in div_results.test_cases.items():
    mins = r.get_min_rmsd()
    print(f"{tc}: {len(r.samples)} samples, min={mins.min():.2f}, mean={mins.mean():.2f}, max={mins.max():.2f}")

INFO:root:Benchmark: toy_benchmark
INFO:root:Mode: diversity
INFO:root:RMSD threshold: 2.0
INFO:root:Samples dir: /global/u1/m/ml5045/confornet/output/demo/diversity/toy_benchmark
INFO:root:Evaluating 2 test cases
Computing metrics:   0%|          | 0/80 [00:00<?, ?it/s]

  mdfA: Min RMSD: 0.71, Mean: 1.28, Max: 2.80



Test cases: 100%|██████████| 2/2 [00:19<00:00,  9.86s/it, fs-4zrb_C-4zrb_H]
INFO:root:
Saved results to /global/u1/m/ml5045/confornet/output/demo/diversity/results.pkl
INFO:root:
=== Overall Summary ===
INFO:root:EvalResults for toy_benchmark (diversity mode)
  Threshold: 2.0 A
  Test cases: 2
  Total samples: 160
  mdfA: 80 samples, min RMSD: 0.71, mean: 1.28
  fs-4zrb_C-4zrb_H: 80 samples, min RMSD: 1.33, mean: 2.00
INFO:root:
Success rate @1: 0.385 +/- 0.216


  fs-4zrb_C-4zrb_H: Min RMSD: 1.33, Mean: 2.00, Max: 3.06
mdfA: 80 samples, min=0.71, mean=1.28, max=2.80
fs-4zrb_C-4zrb_H: 80 samples, min=1.33, mean=2.00, max=3.06


In [23]:
from confornet.eval.metrics import benchmark_success_rate
for k in (10, 20):
    m, s = benchmark_success_rate(div_results, k=k, n_bootstrap=100)
    print(f"  Success @k={k}: {m:.3f} +/- {s:.3f}")

  Success @k=10: 0.868 +/- 0.125
  Success @k=20: 0.948 +/- 0.102


## 2. Visualize diversity samples

Two panels: ref `i` (red) + its closest sample, then ref `j` (blue) + its closest sample.

In [27]:
TC = "fs-4zrb_C-4zrb_H"
tc = div_results.test_cases[TC]
show_best(tc, ref_idx=0, ref_color="#d62728", sample_color="#888").show()
show_best(tc, ref_idx=1, ref_color="#1f77b4", sample_color="#888").show()

fs-4zrb_C-4zrb_H / 4ZRB_C: best sample fs-4zrb_C-4zrb_H_step_5_confornet_0_sample_3.cif (RMSD 1.33 Å)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

fs-4zrb_C-4zrb_H / 4ZRB_H: best sample fs-4zrb_C-4zrb_H_step_5_confornet_1_sample_2.cif (RMSD 1.76 Å)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Evaluate the transfer run

Applies the pretrained `TM_0287v2/6QV1_B` ConforNet to the `mdfA` query and samples. A good outcome is bias toward outward conformation (`6GV1_A`).

In [25]:
xfer_pkl = XFER_OUT / "results.pkl"
run_eval(XFER_OUT, "transfer", xfer_pkl)
xfer_results = load_results(xfer_pkl)
for tc, r in xfer_results.test_cases.items():
    mins = r.get_min_rmsd()
    per_ref = r.rmsd.min(axis=0)
    print(f"{tc}: n={len(r.samples)}, min={mins.min():.2f}, mean={mins.mean():.2f}; per-ref min: " +
          ", ".join(f"{name}={v:.2f}" for name, v in zip(r.references, per_ref)))

for k in (1, 5, 10):
    m, s = benchmark_success_rate(xfer_results, k=k, n_bootstrap=100)
    print(f"  transfer @k={k}: {m:.3f} +/- {s:.3f}")

INFO:root:Benchmark: toy_benchmark
INFO:root:Mode: transfer
INFO:root:RMSD threshold: 2.0
INFO:root:Samples dir: /global/u1/m/ml5045/confornet/output/demo/transfer/toy_benchmark
INFO:root:Evaluating 2 test cases
Test cases: 100%|██████████| 2/2 [00:04<00:00,  2.32s/it, fs-4zrb_C-4zrb_H]
INFO:root:
Saved results to /global/u1/m/ml5045/confornet/output/demo/transfer/results.pkl
INFO:root:
=== Overall Summary ===
INFO:root:EvalResults for toy_benchmark (transfer mode)
  Threshold: 2.0 A
  Test cases: 1
  Total samples: 10
  mdfA: 10 samples, min RMSD: 0.77, mean: 0.95
INFO:root:
Success rate @1: 0.510 +/- 0.381


  mdfA: Min RMSD: 0.77, Mean: 0.95, Max: 1.11
  No CIF files found for fs-4zrb_C-4zrb_H, skipping
mdfA: n=10, min=0.77, mean=0.95; per-ref min: 6VS1_A=0.94, 6GV1_A=0.77
  transfer @k=1: 0.450 +/- 0.357
  transfer @k=5: 0.995 +/- 0.050
  transfer @k=10: 1.000 +/- 0.000


## 4. Visualize transfer samples on mdfA

Outward reference (blue) + the transfer sample closest to it (green).

In [26]:
TC = "mdfA"
tc = xfer_results.test_cases[TC]
show_best(tc, ref_idx=1, ref_color="#1f77b4", sample_color="#2ca02c").show()

mdfA / 6GV1_A: best sample sample_5.cif (RMSD 0.77 Å)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.